# 07. Baseline: TF-IDF + LogisticRegression

Мультиклассовая классификация бренда по тексту запроса.

Метрики: Accuracy, F1-macro/micro, confusion matrix.


In [ ]:
import sys
from pathlib import Path
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (8, 4)


In [ ]:
import joblib
import pandas as pd
import pyarrow.parquet as pq
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, ConfusionMatrixDisplay

DATA = ROOT / 'файлы' / 'query_clicks.parquet'
MODELS = ROOT / 'models'
MODELS.mkdir(exist_ok=True)
FIG = ROOT / 'figures'
FIG.mkdir(exist_ok=True)


In [ ]:
pf = pq.ParquetFile(DATA)
parts = []
seen = 0
for i in range(pf.num_row_groups):
    t = pf.read_row_group(i, columns=['toValidUTF8(query_text)', 'toValidUTF8(sku_brand_name)'])
    df = t.to_pandas()
    df.columns = ['query', 'brand']
    parts.append(df)
    seen += len(df)
    if seen >= 150000:
        break
df = pd.concat(parts, ignore_index=True)
df['query'] = df['query'].fillna('').astype(str).str.strip()
df['brand'] = df['brand'].fillna('').astype(str).str.strip()
df = df[(df['query'].str.len() >= 2) & (df['brand'].str.len() >= 2)].drop_duplicates('query')
top = {b for b, _ in Counter(df['brand']).most_common(80)}
df = df[df['brand'].isin(top)]
print(df.shape, 'classes', df['brand'].nunique())


In [ ]:
Xtr, Xte, ytr, yte = train_test_split(
    df['query'], df['brand'], test_size=0.2, random_state=42, stratify=df['brand']
)
pipe = Pipeline([
    ('tfidf', TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4), min_df=2, max_features=50000)),
    ('clf', LogisticRegression(max_iter=200, solver='saga', n_jobs=-1)),
])
pipe.fit(Xtr, ytr)
pred = pipe.predict(Xte)
print('accuracy', (pred == yte).mean())
print('f1_macro', f1_score(yte, pred, average='macro'))
print('f1_micro', f1_score(yte, pred, average='micro'))
print(classification_report(yte, pred, zero_division=0)[:2000])
joblib.dump(pipe, MODELS / 'brand_clf.joblib')


In [ ]:
top15 = [b for b, _ in Counter(yte).most_common(15)]
mask = yte.isin(top15)
fig, ax = plt.subplots(figsize=(10, 8))
ConfusionMatrixDisplay.from_predictions(
    yte[mask], pred[mask], labels=top15, ax=ax, xticks_rotation=45, colorbar=False
)
ax.set_title('Confusion matrix — бренды (top-15)')
fig.tight_layout()
fig.savefig(FIG / '13_confusion_matrix.png', dpi=140)
plt.show()
